# Pipeline interativo em CCTA externo: OrCaScore e MM-WHS

## Objetivo

Executar um exame externo etapa por etapa com os mesmos componentes do pipeline do `main.ipynb`. A constante `DATASET` seleciona `"orcascore"` ou `"mmwhs"`; `SUBSET` e `EXAM_ID` escolhem o caso.

O OrCaScore recebe a inversão visual do eixo 1 validada contra o ImageCAS, sem alterar a ordem das fatias. O MM-WHS mantém seu layout nativo. Os bancos não fornecem a máscara coronariana binária usada pelo projeto; portanto, o notebook detecta e segmenta, mas não calcula Dice nem acerto dos óstios.

In [ ]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    if (candidate / "pyproject.toml").is_file():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Raiz do projeto não encontrada.")

src_dir = REPO_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.project.notebook_env import configure_notebook_environment

REPO_ROOT = configure_notebook_environment()

In [ ]:
import numpy as np

from utils.project.ccta_datasets import (
    align_ccta_volume_to_imagecas_view,
    discover_ccta_dataset,
    load_ccta_volume,
)
from utils.project.notebook_env import (
    load_notebook_pipeline_config,
    resolve_existing_path,
)
from utils.segmentation.artery_segmentation import normal_region_growing_from_ostia
from utils.segmentation.pipeline_arteries import get_artery_postprocessing_stages
from utils.segmentation.pipeline_detection import (
    detect_ostia,
    filter_located_aorta_circles,
    locate_aorta_circles,
    segment_aorta,
)
from utils.segmentation.pipeline_orchestration import summarize_aorta_circles
from utils.segmentation.pipeline_preprocessing import (
    compute_vesselness,
    preprocess_ccta_volume,
)
from utils.visualization.image_slices import plot_slices, visualize_circles_on_slices
from utils.visualization.preprocessing_views import plot_pipeline_preprocessing_stages
from utils.visualization.vesselness import display_vesselness_summary
from utils.visualization.volume import visualize_3d_k3d, visualize_aorta_ostia_artery

## Configuração

Altere as constantes abaixo. Com `EXAM_ID = None`, o notebook escolhe o exame com quantidade de fatias mais próxima da mediana do subset selecionado.

In [ ]:
DATASET = "orcascore"  # "orcascore" ou "mmwhs"
SUBSET = "train"  # "train" ou "test"
EXAM_ID = None  # Exemplos: "TEV1P1" ou "ct_train_1001"
RESOLUTION = "mid"  # "mid" ou "high"
ALIGN_ORCASCORE_TO_IMAGECAS_VIEW = True
CONFIG_FILE = REPO_ROOT / "config/pipeline_config.json"

DATASET_SPECS = {
    "orcascore": {
        "env": "ORCASCORE_BASE_PATH",
        "candidates": [Path("/media/matheus/HD/DatasetsCCTA/Orca_Score_Calcium")],
        "name": "OrCaScore",
    },
    "mmwhs": {
        "env": "MMWHS_BASE_PATH",
        "candidates": [Path("/media/matheus/HD/DatasetsCCTA/MM-WHS-2017-Dataset")],
        "name": "MM-WHS 2017",
    },
}

DATASET = DATASET.lower()
if DATASET not in DATASET_SPECS:
    raise ValueError("DATASET deve ser 'orcascore' ou 'mmwhs'.")
if SUBSET not in {"train", "test"}:
    raise ValueError("SUBSET deve ser 'train' ou 'test'.")
if RESOLUTION not in {"mid", "high"}:
    raise ValueError("RESOLUTION deve ser 'mid' ou 'high'.")

In [ ]:
spec = DATASET_SPECS[DATASET]
base_path = resolve_existing_path(
    spec["env"],
    spec["candidates"],
    spec["name"],
)
inventory = discover_ccta_dataset(DATASET, base_path)
subset_inventory = inventory.loc[inventory["subset"].eq(SUBSET)].copy()
if subset_inventory.empty:
    raise ValueError(f"Nenhum exame encontrado no subset {SUBSET!r}.")

if EXAM_ID is None:
    median_slices = subset_inventory["slice_count"].median()
    record = subset_inventory.loc[
        (subset_inventory["slice_count"] - median_slices).abs().idxmin()
    ]
else:
    matches = subset_inventory.loc[subset_inventory["exam_id"].eq(str(EXAM_ID))]
    if matches.empty:
        examples = ", ".join(subset_inventory["exam_id"].head(8))
        raise ValueError(f"EXAM_ID {EXAM_ID!r} não encontrado. Exemplos: {examples}")
    record = matches.iloc[0]

CONFIG = load_notebook_pipeline_config(CONFIG_FILE, RESOLUTION)
USE_GPU = bool(CONFIG.get("USE_GPU", False))
backend = "gpu" if USE_GPU else "cpu"

print(f"Dataset/subset: {record['dataset']} / {record['subset']}")
print(f"Exame: {record['exam_id']}")
print(f"Arquivo: {record['path']}")
print(f"Resolução: {RESOLUTION}")
print(f"Backend solicitado: {backend}")
print(f"Orientação informada: {record['reported_orientation']}")

## Carregamento e alinhamento visual

O OrCaScore é invertido somente no eixo 1 para remover o espelhamento observado contra o ImageCAS. A transformação não interpola os dados e preserva intensidades, espaçamento, dimensões e a ordem axial das fatias.

In [ ]:
native_image = load_ccta_volume(record).astype(np.float32, copy=False)
spacing = (
    float(record["spacing_x_mm"]),
    float(record["spacing_y_mm"]),
    float(record["spacing_z_mm"]),
)

if ALIGN_ORCASCORE_TO_IMAGECAS_VIEW:
    img, flipped_axes = align_ccta_volume_to_imagecas_view(
        native_image,
        str(record["dataset"]),
    )
else:
    img = native_image
    flipped_axes = ()

print(f"Shape: {img.shape}")
print(f"Espaçamento (x, y, z): {spacing}")
print(f"Orientação informada preservada: {record['reported_orientation']}")
print(f"Eixos invertidos visualmente: {flipped_axes or 'nenhum'}")

preview_slices = np.linspace(0, img.shape[2] - 1, num=min(5, img.shape[2]), dtype=int)
plot_slices(
    img, preview_slices.tolist(), title=f"{record['dataset']} — {record['exam_id']}"
)

## Pré-processamento

A imagem passa pelo mesmo downsampling, threshold HU e maior componente conectada usados no pipeline principal.

In [ ]:
image_data = preprocess_ccta_volume(
    img,
    spacing,
    CONFIG,
    include_intermediates=True,
)

down_image = image_data["down_image"]
threshold_mask = image_data["threshold_mask"]
lcc_image = image_data["lcc_image"]
lcc_mask = image_data["lcc_mask"]
downscale_factors = image_data["downscale_factors"]
scaled_spacing = image_data["scaled_spacing"]
visual_spacing = (scaled_spacing[1], scaled_spacing[0], scaled_spacing[2])
preprocessing_details = image_data["preprocessing_details"]

print(f"Shape original: {img.shape}")
print(f"Shape processado: {lcc_image.shape}")
print(f"Fatores de downscale: {tuple(downscale_factors)}")
print(f"Espaçamento processado (dx, dy, dz): {scaled_spacing}")
print(f"Modo de threshold: {preprocessing_details.get('threshold_mode')}")
print(
    "Limiares HU: "
    f"{preprocessing_details.get('min_threshold')} a "
    f"{preprocessing_details.get('max_threshold')}"
)
print(f"Voxels candidatos: {int(threshold_mask.sum()):,}")
print(f"Voxels na LCC: {int(lcc_mask.sum()):,}")

In [ ]:
slice_idx = lcc_image.shape[2] // 2
plot_pipeline_preprocessing_stages(image_data, slice_idx)

In [ ]:
sample_slices = np.linspace(
    0,
    lcc_image.shape[2] - 1,
    num=min(5, lcc_image.shape[2]),
    dtype=int,
).tolist()
plot_slices(lcc_image, sample_slices, title="LCC em fatias distribuídas no volume")

## Localização e segmentação da aorta

Os círculos são localizados, filtrados pelo método configurado e usados para inicializar o level set. O envelope de trajetória é aplicado quando definido em `LEVEL_SET`.

In [ ]:
raw_circles = locate_aorta_circles(
    lcc_image,
    downscale_factors,
    scaled_spacing,
    CONFIG["CIRCLE_DETECTION"],
)
if not raw_circles:
    raise RuntimeError("Nenhum círculo da aorta foi detectado para este exame.")

detected_circles, circle_filter_details = filter_located_aorta_circles(
    raw_circles,
    scaled_spacing,
    lcc_image.shape[2],
    CONFIG["CIRCLE_DETECTION"],
)
if not detected_circles:
    raise RuntimeError("O filtro removeu todos os círculos da aorta.")

circle_summary = summarize_aorta_circles(detected_circles, lcc_image.shape[2])
print(f"Círculos antes/depois do filtro: {len(raw_circles)} / {len(detected_circles)}")
print(f"Método do filtro: {circle_filter_details.get('aorta_circle_filter_method')}")
print(f"Círculos totais: {circle_summary['aorta_circle_count']}")
print(
    f"Detectados: {circle_summary['aorta_detected_circle_count']} | "
    f"interpolados: {circle_summary['aorta_interpolated_circle_count']}"
)

visualize_circles_on_slices(
    lcc_image,
    detected_circles,
    num_samples=6,
    vmin=float(lcc_image.min()),
    vmax=float(lcc_image.max()),
)

In [ ]:
aorta_mask = segment_aorta(
    lcc_image,
    detected_circles,
    CONFIG["LEVEL_SET"],
    use_gpu=USE_GPU,
).astype(np.uint8)
if not np.any(aorta_mask):
    raise RuntimeError("A segmentação da aorta produziu uma máscara vazia.")

print(f"Voxels na máscara da aorta: {int(aorta_mask.sum()):,}")
aorta_plot = visualize_3d_k3d(
    aorta_mask,
    spacing=visual_spacing,
    color=0xFF5555,
    opacity=0.45,
)

## Mapas de vesselness

São calculados mapas separados para localizar os óstios e conduzir a segmentação arterial.

In [ ]:
vesselness_ostia = compute_vesselness(
    lcc_image,
    vesselness_config=CONFIG["VESSELNESS_AORTA"],
    use_gpu=USE_GPU,
)
display_vesselness_summary(
    vesselness_ostia,
    label="Vesselness dos óstios",
    title="Vesselness para detecção dos óstios",
)

In [ ]:
vesselness_artery = compute_vesselness(
    lcc_image,
    vesselness_config=CONFIG["VESSELNESS_ARTERY"],
    use_gpu=USE_GPU,
)
display_vesselness_summary(
    vesselness_artery,
    label="Vesselness arterial",
    title="Vesselness para segmentação arterial",
)

## Detecção dos óstios

A detecção usa a superfície da aorta e o vesselness. Sem uma máscara coronariana compatível, as coordenadas podem ser inspecionadas, mas não classificadas como corretas ou toleráveis.

In [ ]:
ostia_left, ostia_right = detect_ostia(
    aorta_mask,
    vesselness_ostia,
    scaled_spacing,
    CONFIG,
)

print(f"Óstio esquerdo: {ostia_left}")
print(f"Óstio direito: {ostia_right}")
print("Avaliação dos óstios: indisponível sem referência coronariana.")

ostia_plot = visualize_aorta_ostia_artery(
    aorta_mask,
    ostia_left,
    ostia_right,
    spacing=visual_spacing,
    plot_name=f"{record['dataset']} {record['exam_id']}: aorta e óstios",
)

## Segmentação arterial por region growing

O crescimento parte dos óstios detectados e usa o mapa arterial de vesselness.

In [ ]:
raw_artery_mask = normal_region_growing_from_ostia(
    vesselness_artery,
    ostia_left,
    ostia_right,
    CONFIG,
).astype(np.uint8)

print(f"Voxels antes da morfologia: {int(raw_artery_mask.sum()):,}")
if not np.any(raw_artery_mask):
    print("Aviso: o region growing produziu uma máscara vazia.")

raw_plot = visualize_aorta_ostia_artery(
    aorta_mask,
    ostia_left,
    ostia_right,
    artery_mask=raw_artery_mask,
    spacing=visual_spacing,
    plot_name=f"{record['dataset']} {record['exam_id']}: artéria sem morfologia",
)

## Pós-processamento morfológico

Fechamento e dilatação usam o mesmo helper e os mesmos parâmetros da configuração efetiva.

In [ ]:
postprocessing_stages = get_artery_postprocessing_stages(raw_artery_mask, CONFIG)
closed_mask = postprocessing_stages["closed_mask"]
artery_mask = postprocessing_stages["final_mask"]

print(f"Sem morfologia: {int(raw_artery_mask.sum()):,} voxels")
print(f"Após fechamento: {int(closed_mask.sum()):,} voxels")
print(f"Após dilatação: {int(artery_mask.sum()):,} voxels")

processed_plot = visualize_aorta_ostia_artery(
    aorta_mask,
    ostia_left,
    ostia_right,
    artery_mask=artery_mask,
    spacing=visual_spacing,
    plot_name=f"{record['dataset']} {record['exam_id']}: artéria após morfologia",
)

## Resumo do caso

O resumo contém apenas medidas observáveis sem referência. Dice, distância ao label e taxa de acerto não são calculados, pois os rótulos desses bancos não representam a mesma tarefa coronariana do ImageCAS.

In [ ]:
voxel_volume_mm3 = float(np.prod(scaled_spacing))
summary = {
    "dataset": record["dataset"],
    "subset": record["subset"],
    "exam_id": record["exam_id"],
    "reported_orientation": record["reported_orientation"],
    "visual_flip_axes": flipped_axes,
    "processed_shape": tuple(int(value) for value in lcc_image.shape),
    "aorta_circle_count": len(detected_circles),
    "aorta_volume_ml": float(aorta_mask.sum() * voxel_volume_mm3 / 1000.0),
    "ostia_left": tuple(int(value) for value in ostia_left),
    "ostia_right": (
        tuple(int(value) for value in ostia_right) if ostia_right is not None else None
    ),
    "artery_volume_before_morphology_ml": float(
        raw_artery_mask.sum() * voxel_volume_mm3 / 1000.0
    ),
    "artery_volume_after_morphology_ml": float(
        artery_mask.sum() * voxel_volume_mm3 / 1000.0
    ),
    "dice": None,
    "ostia_accuracy": None,
}

for key, value in summary.items():
    print(f"{key}: {value}")